# Notebook 09 — Cross-plant transfer between SWaT and WADI

*Corrected version of the original transfer notebook `notebook46c429edf0 (1).ipynb` (called
"Notebook 09" inside it). It also does the training part of the original
`10-transfer-significance (1).ipynb`; Notebook_10 now only analyses the saved results.*

**What this notebook does.** Each plant is projected to a common 32-dimensional space. MAML
and a conventional autoencoder are trained on the source plant, adapted with K normal windows
of the target plant (K = 20, 50, 100), and scored on the target's attack recording. This is
repeated for 6 training seeds and both directions.

**Why.** Transfer was the one setting where the old paper saw a possible MAML advantage. The
original fitted the target plant's scaling and PCA on the target's **entire** normal
recording, which a few-shot method would not have. That leak is removed here.

**Input.** The output of Notebook_03 for both plants, and this repository folder.

**Output.** `transfer_seed<seed>.json` per training seed and model checkpoints. Expected time
on a Kaggle GPU: about 25 minutes per training seed for both directions (up to 5,000 outer
steps each, with early stopping), so about 2.5 hours in total.

### What was corrected
1. **Leak fix.** In the main ("leak-free") view the target's MinMax scaling, PCA and clipping
   ranges are fitted on the K support windows only (K x 30 rows; 600 rows at K = 20), built
   from the unscaled target data. The source plant still uses its full normal data, which is
   allowed. The original, leaky projection is kept as a second view, labelled legacy, so the
   old numbers can be lined up.
2. **"Scratch" is now a real scratch model:** a fresh LSTM-AE trained only on the K support
   windows (Adam, 300 full-batch steps). The original "Scratch" was a random network given 10
   SGD steps, essentially untrained. It is kept as "Scratch (legacy)".
3. **Six training seeds for all K** (the original seeded run used K = 50 only and one-sided
   tests; the single-run table used one training run).
4. **Zero-step controls** for MAML-transfer and Static-transfer, and an untrained floor.
5. **Target-static** (trained on the target's full normal data) is reported as an upper
   reference, not as a few-shot method.
6. PCA explained variance is recorded for the full-data fit and for every support-only fit.
7. Label-free F1 is reported next to the oracle best-F1.

**Kept as in the original seeded run:** PCA to 32 dimensions with min-max scaling and
clipping; FOMAML inner SGD 0.01 x 10, Adam 0.001, 4 tasks, 20/20, clipping 1.0, up to
5,000 outer steps, validation every 250 steps, early stopping after 6 checks without
improvement; conventional recipe with 120 epochs and patience 12; adaptation 10 steps at 0.01.

In [ ]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import json, time, numpy as np, torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| SMOKE TEST (numbers are meaningless)" if SMOKE else "")

## 1 — Configuration

In [ ]:
CFG = dict(directions=[["wadi", "swat"], ["swat", "wadi"]], train_seeds=[42, 123, 456, 789, 1024, 2048],
           support_seeds=[0, 1, 2], k_shots=[20, 50, 100], dim=32,
           n_outer=5000, val_every=250, patience=6, use_scheduler=False, inner_lr=0.01, inner_steps=10,
           outer_lr=1e-3, tasks_per_batch=4, support_size=20, query_size=20, val_episodes_per_task=16,
           val_seed=2024, static_max_epochs=120, static_patience=12, adapt_steps=10, adapt_lr=0.01,
           scratch_steps=300, scratch_lr=1e-3, legacy_scratch_steps=10)
if SMOKE:
    CFG.update(train_seeds=[42], support_seeds=[0], k_shots=[20], n_outer=4, val_every=2, patience=2,
               val_episodes_per_task=1, static_max_epochs=1, static_patience=1, scratch_steps=5)
TRAIN_SEEDS = CFG["train_seeds"]
print(CFG)

## 2 — Data and the full-data projections

For each plant, the full-data projection (MinMax, PCA to 32, min-max, clipping, all fitted on
that plant's normal data) is what the plant uses **as a source**. Used on a target, the same
projection is the original's leaky set-up.

In [ ]:
P = {p: sc.load_plant(p) for p in ["swat", "wadi"]}
full = {}
for p, d in P.items():
    proj = sc.fit_projection(d["normal_rows_unscaled"], CFG["dim"])
    nw = sc.project_windows(proj, d["normal_unscaled"])
    full[p] = {"proj": proj, "normal": nw, "attack": sc.project_windows(proj, d["attack_unscaled"]),
               "tasks": {g: nw[d["normal_regime"] == g] for g in d["split"]["meta_train"]},
               "val": sc.fixed_episodes({g: nw[d["normal_regime"] == g] for g in d["split"]["meta_val"]},
                                        CFG["val_episodes_per_task"], CFG["val_seed"])}
    evr = proj["explained_variance_ratio"]
    print(f"{p}: variance kept by 32 components {proj['explained_variance']:.4%}; PC1 {evr[0]:.2%}, "
          f"first 5 {evr[:5].sum():.2%}, first 10 {evr[:10].sum():.2%}; normal windows {nw.shape}")

## 3 — Finding earlier results and checkpoints (for resuming)

In [ ]:
def find_all(name):
    hits = [os.path.join(OUT, name)] if os.path.exists(os.path.join(OUT, name)) else []
    if os.path.isdir("/kaggle/input"):
        hits += [os.path.join(d, name) for d, _, f in os.walk("/kaggle/input") if name in f]
    return hits

def latest_checkpoint(name):
    best, best_step = None, -1
    for p in find_all(name):
        try:
            s = torch.load(p, map_location="cpu", weights_only=False).get("step", 0)
        except Exception:
            continue
        if s > best_step:
            best, best_step = p, s
    return best

def result_name(seed):
    return f"transfer_seed{seed}{'_SMOKE' if SMOKE else ''}.json"

## 4 — Training for one seed

One MAML model per source plant (meta-trained on its regimes), one conventional
autoencoder per plant (trained on all its normal windows). The conventional model of the
source is the Static-transfer starting point; the conventional model of the target is the
Target-static upper reference.

In [ ]:
def train_models(seed):
    models, info = {}, {}
    for p in ["swat", "wadi"]:
        name = f"transfer_maml_{p}_seed{seed}"
        sc.seed_everything(seed)
        m = sc.LSTMAutoencoder(CFG["dim"]).to(DEVICE)
        hit = find_all(f"{name}_best.pt")
        if hit:
            ck = torch.load(hit[0], map_location=DEVICE, weights_only=False)
            m.load_state_dict(ck["model_state_dict"]); info[f"maml_{p}"] = ck["info"]
        else:
            m, info[f"maml_{p}"] = sc.train_maml(
                m, full[p]["tasks"], full[p]["val"], DEVICE, seed=seed, n_outer=CFG["n_outer"],
                val_every=CFG["val_every"], patience=CFG["patience"], inner_lr=CFG["inner_lr"],
                inner_steps=CFG["inner_steps"], outer_lr=CFG["outer_lr"],
                tasks_per_batch=min(CFG["tasks_per_batch"], len(full[p]["tasks"])),
                use_scheduler=CFG["use_scheduler"], ckpt_path=os.path.join(OUT, f"{name}_ckpt.pt"),
                resume_from=latest_checkpoint(f"{name}_ckpt.pt"))
            torch.save({"model_state_dict": m.state_dict(), "info": info[f"maml_{p}"]}, os.path.join(OUT, f"{name}_best.pt"))
        models[f"maml_{p}"] = m
        fname = f"transfer_static_{p}_seed{seed}.pt"
        sc.seed_everything(seed)
        s = sc.LSTMAutoencoder(CFG["dim"]).to(DEVICE)
        hit = find_all(fname)
        if hit:
            ck = torch.load(hit[0], map_location=DEVICE, weights_only=False)
            s.load_state_dict(ck["model_state_dict"]); info[f"static_{p}"] = ck["info"]
        else:
            s, info[f"static_{p}"] = sc.train_conventional(s, full[p]["normal"], DEVICE, seed=seed,
                                                           max_epochs=CFG["static_max_epochs"],
                                                           patience=CFG["static_patience"])
            torch.save({"model_state_dict": s.state_dict(), "info": info[f"static_{p}"]}, os.path.join(OUT, fname))
        models[f"static_{p}"] = s
    return models, info

## 5 — Scoring for one seed

For every direction, K and support seed, the same K target windows are used by every
method, in both views. In the leak-free view the target projection is refitted on those K
windows each time.

In [ ]:
def score(model, sup, att, y, steps, lr=None):
    a = sc.inner_adapt(model, sc.to_tensor(sup, DEVICE), lr or CFG["adapt_lr"], steps) if steps else model
    tau = sc.label_free_threshold(sc.window_errors(a, sup, DEVICE))
    return sc.detection_metrics(sc.window_errors(a, att, DEVICE), y, tau)

def evaluate(models, seed):
    res = {}
    for src, tgt in CFG["directions"]:
        T = P[tgt]
        for k in CFG["k_shots"]:
            for s in CFG["support_seeds"]:
                idx = np.sort(np.random.RandomState([s, k]).choice(len(T["normal_unscaled"]), k, replace=False))
                sup_u = T["normal_unscaled"][idx]
                proj = sc.fit_projection(sup_u.reshape(-1, sup_u.shape[2]), CFG["dim"])
                views = {"leak_free": (sc.project_windows(proj, sup_u), sc.project_windows(proj, T["attack_unscaled"])),
                         "legacy_full_target_fit": (full[tgt]["normal"][idx], full[tgt]["attack"])}
                entry = {"target_projection_explained_variance": proj["explained_variance"]}
                for view, (sup, att) in views.items():
                    r = {}
                    r["MAML-transfer"] = score(models[f"maml_{src}"], sup, att, T["any"], CFG["adapt_steps"])
                    r["MAML-transfer (0 steps)"] = score(models[f"maml_{src}"], sup, att, T["any"], 0)
                    r["Static-transfer"] = score(models[f"static_{src}"], sup, att, T["any"], CFG["adapt_steps"])
                    r["Static-transfer (0 steps)"] = score(models[f"static_{src}"], sup, att, T["any"], 0)
                    torch.manual_seed(seed * 1000 + s)
                    scratch = sc.train_on_support(sc.LSTMAutoencoder(CFG["dim"]).to(DEVICE), sup, DEVICE,
                                                  seed=seed * 1000 + s, steps=CFG["scratch_steps"], lr=CFG["scratch_lr"])
                    r["Scratch"] = score(scratch, sup, att, T["any"], 0)
                    torch.manual_seed(seed * 1000 + s)
                    r["Scratch (legacy)"] = score(sc.LSTMAutoencoder(CFG["dim"]).to(DEVICE), sup, att, T["any"],
                                                  CFG["legacy_scratch_steps"])
                    torch.manual_seed(seed * 1000 + s + 7)
                    r["LSTM-AE untrained (floor)"] = score(sc.LSTMAutoencoder(CFG["dim"]).to(DEVICE), sup, att, T["any"], 0)
                    if view == "legacy_full_target_fit":
                        r["Target-static (upper reference)"] = score(models[f"static_{tgt}"], sup, att, T["any"], 0)
                    entry[view] = r
                res[f"{src}->{tgt}|{k}|{s}"] = entry
            print(f"  seed {seed}: {src}->{tgt} K={k} scored")
    return res

## 6 — Run all training seeds (finished seeds are skipped)

In [ ]:
for seed in TRAIN_SEEDS:
    if find_all(result_name(seed)):
        print(f"seed {seed}: result exists, skipping"); continue
    t0 = time.time()
    models, info = train_models(seed)
    res = evaluate(models, seed)
    sc.save_json(os.path.join(OUT, result_name(seed)),
                 {"experiment": "SWaT <-> WADI transfer, corrected", "smoke_test": SMOKE, "train_seed": seed,
                  "config": CFG, "full_fit_explained_variance": {p: full[p]["proj"]["explained_variance"] for p in full},
                  "full_fit_variance_ratio": {p: full[p]["proj"]["explained_variance_ratio"] for p in full},
                  "training": info, "results": res, "device": str(DEVICE), "torch": torch.__version__,
                  "wall_seconds": time.time() - t0})
    print(f"seed {seed}: saved {result_name(seed)} ({time.time() - t0:.0f}s)")